# NicheBridge kidney tutorial

This notebook presents six kidney analysis sections and a freshly fitted NicheBridge demonstration. The supplied full dataset contains **31,745 spots and 3,000 genes**. Its domain labels, latent coordinates, UMAP coordinates and reconstructed expression are **precomputed inputs**, not outputs of the new fit.

Python plotting and downstream statistical helpers live only in `tutorials/kidney_workflow.py`; they are not modeling-package APIs. All figures, tables, parameters, input hashes and software versions are saved under `results/kidney_tutorial`. The supplied-data analyses and fresh model outputs have separate output folders.

Run cells from top to bottom. The local environment needs NicheBridge, Scanpy, Matplotlib, SciPy, pandas, GSEApy, statsmodels and Jupyter. The cached Hallmark GMT avoids network access during execution.

In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if
                     (p / "pyproject.toml").is_file() and (p / "tutorials").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the NicheBridge repository or its tutorials folder.")
DEFAULT_INPUT = PROJECT_ROOT / "data" / "kidney" / "adata_all_rec.h5ad"
if not DEFAULT_INPUT.is_file() and (PROJECT_ROOT / "adata_all_rec.h5ad").is_file():
    DEFAULT_INPUT = PROJECT_ROOT / "adata_all_rec.h5ad"
INPUT_H5AD = Path(os.environ.get("NICHEBRIDGE_KIDNEY_DATA", DEFAULT_INPUT))
OUTPUT_ROOT = Path(os.environ.get("NICHEBRIDGE_TUTORIAL_OUTPUT",
                                 PROJECT_ROOT / "results" / "kidney_tutorial"))
REPRODUCTION_DIR = OUTPUT_ROOT / "supplied_data_analysis"
HALLMARK_GMT = PROJECT_ROOT / "tutorials" / "resources" / "MSigDB_Hallmark_2020.gmt"

SEED = 42
ANALYSIS_LAYER = "log1p"
ANALYSIS_SCALE = "log1p"
N_PERMUTATIONS = 1000
RUN_FRESH_FIT = True

# Tutorial helpers are repository-local; install NicheBridge before running.
helper_path = str(PROJECT_ROOT / "tutorials")
if helper_path not in sys.path:
    sys.path.insert(0, helper_path)

import nichebridge as nb
import pandas as pd
from IPython.display import Image, display
import kidney_workflow as kw

if not INPUT_H5AD.is_file():
    raise FileNotFoundError(f"Input file missing: {INPUT_H5AD}. See data/README.md.")
if not HALLMARK_GMT.is_file():
    raise FileNotFoundError(f"Local gene sets missing: {HALLMARK_GMT}")
print("NicheBridge:", nb.__version__)
print("Input:", INPUT_H5AD)
print("Outputs:", OUTPUT_ROOT)


## Explicit input fields and provenance

Differential expression, GSEA and marker profiles use the supplied **`log1p` layer unchanged**: it contains log1p of raw UMI counts, without library-size normalization. Its name is selected explicitly; the notebook never guesses transformation from `uns['log1p']` or normalizes/logs this matrix again. All transformations are explicit and recorded with the outputs.

To replay the supplied `X` matrix unchanged, explicitly set `ANALYSIS_LAYER=None` and `ANALYSIS_SCALE="supplied"` above. Its upstream numerical scale is not independently documented; Scanpy's reported log-fold changes assume a log-expression convention.

The loader maps supplied domain, latent, UMAP and reconstruction fields to `supplied_domain`, `X_supplied_latent`, `X_supplied_umap` and `supplied_reconstruction`. These names indicate provenance. Missing marker genes are reported and never filled with invented values.

In [ ]:
adata = kw.load_supplied_dataset(
    INPUT_H5AD, REPRODUCTION_DIR, seed=SEED,
    analysis_layer=ANALYSIS_LAYER, analysis_scale=ANALYSIS_SCALE,
)
info = adata.uns["nichebridge_tutorial"]
display(pd.Series({key: info[key] for key in ["n_spots", "n_genes", "analysis_layer", "analysis_scale", "input_sha256", "representation_status"]}))
display(pd.DataFrame({"gene": kw.MARKER_GENES, "available": [gene in adata.var_names for gene in kw.MARKER_GENES]}))
section_results = {}

## Spatial domains across age and sex

Eight tissue panels show the supplied domain annotations at W3, W12, W52 and W92, with females above males. The section orientation follows the supplied figure convention. Text labels replace decorative external image files.

In [ ]:
section_results["domains"] = kw.plot_domain_maps(adata, REPRODUCTION_DIR)
kw.show_figures(section_results["domains"])

## Sex-stratified age-associated differential expression

Each supplied domain is tested for W92 versus W12 separately within female and male samples using the Wilcoxon rank-sum test. The thresholds are: at least five spots per age group, absolute reported log-fold change ≥ 0.1, and adjusted P ≤ 1e-20. Upregulated and downregulated genes appear above and below zero. The complete gene tables and group counts are saved, including any skipped comparison.

In [ ]:
section_results["de"] = kw.age_de_by_domain(adata, REPRODUCTION_DIR)
display(pd.read_csv(section_results["de"]["tables"]["counts"]))
kw.show_figures(section_results["de"])

## Proximal-tubule representation across ages

The supplied annotations identify domain 1 as PT(S1/S2) and domain 6 as PT(S3). Three panels reuse the supplied UMAP coordinates and show domain, sex and age on the same axes. Female–male centroid distances are calculated in the supplied **16-dimensional latent representation**, not the two-dimensional UMAP. These are descriptions of the supplied representation, not a fresh fit.

In [ ]:
section_results["latent"] = kw.plot_proximal_latent(adata, REPRODUCTION_DIR)
display(pd.read_csv(section_results["latent"]["tables"]["centroids"]))
kw.show_figures(section_results["latent"])

## Sex-stratified Hallmark pathway enrichment in PT(S3)

Within domain 6, genes are ranked by W92-versus-W12 Wilcoxon scores for each sex. Preranked GSEA uses all 50 cached Hallmark 2020 gene sets, 1,000 permutations, gene-set sizes 10–500, and seed 42. Input symbols are matched in uppercase; this is case matching rather than inferred cross-species orthology. Duplicate symbols introduced by case matching raise an error.

The figure shows oxidative phosphorylation, fatty-acid metabolism, xenobiotic metabolism, interferon-gamma response and complement. Bar widths are absolute NES; signed values and FDR annotations are displayed. A selected term that cannot be tested is labeled explicitly. Full results, rankings and the GMT hash are saved.

In [ ]:
section_results["hallmark"] = kw.run_hallmark(
    adata, REPRODUCTION_DIR, HALLMARK_GMT,
    permutations=N_PERMUTATIONS, seed=SEED,
)
display(pd.read_csv(section_results["hallmark"]["tables"]["selected_pathways"])[["Term", "sex", "NES", "FDR q-val"]])
kw.show_figures(section_results["hallmark"])

## Marker profiles and age/sex trajectories in PT(S3)

The marker panel is restricted to genes present in the supplied input; its exact present/missing lists are recorded. Expression means across F/M × W12/W52/W92 are transformed to gene-wise Z scores and ordered by average-linkage hierarchical clustering.

The additive model `expression ~ age_num + sex_num` is fitted to spots. Age and Sex sidebars show −log10 of the unadjusted coefficient P values; grey bars show their arithmetic mean capped at 280. These are descriptive spot-level associations. Representative Prlr, Acsm3 and Cyp7b1 group-mean trajectories are also shown. The full dataset remains available for subsequent sections.

In [ ]:
section_results["markers"] = kw.plot_marker_profiles(adata, REPRODUCTION_DIR)
print("Available marker genes:", len(section_results["markers"]["summary"]["present_genes"]))
print("Missing marker genes:", section_results["markers"]["summary"]["missing_genes"])
kw.show_figures(section_results["markers"])

## Cyp7b1 distributions and tissue maps

This section uses **the supplied precomputed reconstruction** mapped from the input reconstruction layer. All spatial domains are retained within the six W12/W52/W92 sections. Female and male distributions are shown alongside tissue maps sharing the 2nd–98th percentile color limits.

The high-expression subsets are explicit: the highest 0.41% for females and 10% for males, with thresholds calculated across the three ages within each sex. Their Spearman correlations are descriptive subset associations; they are not correlations across all spots. Thresholds, selected counts and correlation results are saved.

In [ ]:
section_results["gene_maps"] = kw.plot_gene_maps(
    adata, REPRODUCTION_DIR, genes=["Cyp7b1"],
    high_fractions={"F": 0.0041, "M": 0.1},
)
display(pd.read_csv(section_results["gene_maps"]["tables"]["correlations"]))
kw.show_figures(section_results["gene_maps"])

## Fresh NicheBridge fit on a balanced kidney subset

This separate demonstration recovers verified integer counts from the supplied logged-count layer and fits a new NicheBridge trajectory model on a reproducible subset balanced across the eight sections. It writes a new checkpoint, latent representation, reverse reconstruction and training figures under `fresh_fit`.

The short subset run demonstrates execution and provenance; it is not a converged full-cohort result. Newly clustered domains do not automatically inherit the biological meanings of supplied domain numbers. The 31,745-spot analyses above remain labeled as supplied-data analyses.

The modeling calls used by the helper are:

```python
config = nb.TrajectoryConfig(pretrain_epochs=20, epochs=40,
    n_top_genes=256, coupling_steps=16, ode_steps=64, seed=SEED)
model = nb.NicheBridgeTrajectory(config)
model.fit({"spatial": training}, slice_key="section")
latent = model.transform("spatial")
reconstructed = model.reconstruct("spatial")
```

Here `training` contains only validated recovered counts, observation/gene identifiers and slice metadata. Biological domain identities are not used to fit or name the fresh clusters.

In [ ]:
fresh = None
if RUN_FRESH_FIT:
    from fit_kidney import run_fresh_fit
    fresh = run_fresh_fit(input_path=INPUT_H5AD, output_dir=OUTPUT_ROOT / "fresh_fit", seed=SEED)
    display(pd.Series(fresh["summary"]))
    for figure in fresh["figures"]:
        if Path(figure).suffix.lower() == ".png":
            display(Image(filename=str(figure)))
else:
    print("Fresh fit explicitly disabled by RUN_FRESH_FIT=False; no new model result is claimed.")

## Saved outputs

Every figure is saved as PNG and PDF; all numerical tables are CSV. Input and section manifests record source fields, parameters, filtering, gene availability and software versions. The fresh fit has its own checkpoint and provenance. Notebook outputs below come only from the current execution.

In [ ]:
import json
from datetime import datetime, timezone

run_summary = {
    "tutorial": "NicheBridge kidney tutorial",
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "input_sha256": info["input_sha256"],
    "supplied_data_sections": {name: result["manifest"] for name, result in section_results.items()},
    "fresh_fit_executed": fresh is not None,
    "fresh_fit_adata": str(fresh["adata"]) if fresh is not None else None,
    "fresh_fit_checkpoint": str(fresh["checkpoint"]) if fresh is not None else None,
}
manifest_path = OUTPUT_ROOT / "NicheBridge_notebook_run.json"
manifest_path.write_text(json.dumps(run_summary, indent=2), encoding="utf-8")
print("Run manifest:", manifest_path)
display(pd.DataFrame([{"section": name, "manifest": result["manifest"]} for name, result in section_results.items()]))